In [26]:
import pandas as pd
import matplotlib.pyplot as plt

In [27]:
year = 1999
df = pd.read_excel('usa_employment_1999.xlsx')[['fipstate', 'fipscty', 'FIPS', 'CZ', 'naics', 'empflag', 'emp']]
df = df.dropna(subset=['CZ'])
df

,fipstate,fipscty,FIPS,CZ,naics,empflag,emp
0,1,1,1001,11101.0,1131,A,0
1,1,1,1001,11101.0,1133,A,0
2,1,1,1001,11101.0,1151,A,0
3,1,1,1001,11101.0,1153,A,0
4,1,1,1001,11101.0,2123,B,0
...,...,...,...,...,...,...,...
433677,56,45,56045,34601.0,8122,A,0
433678,56,45,56045,34601.0,8123,A,0
433679,56,45,56045,34601.0,8131,A,0
433680,56,45,56045,34601.0,8134,A,0


In [28]:
df.describe()


,fipstate,fipscty,FIPS,CZ,naics,emp
count,432066.000000,432066.000000,432066.00000,432066.000000,432066.000000,432066.000000
mean,30.234841,99.914633,30334.75607,19797.771498,4787.777379,208.504252
std,15.189385,108.596413,15208.81891,11158.649123,1636.991559,1438.248728
min,1.000000,1.000000,1001.00000,100.000000,1131.000000,0.000000
25%,18.000000,33.000000,18129.00000,10700.000000,3344.000000,0.000000
50%,29.000000,75.000000,29165.00000,19903.000000,4541.000000,0.000000
75%,44.000000,127.000000,44003.00000,29302.000000,5619.000000,54.000000
max,56.000000,840.000000,56045.00000,39400.000000,8139.000000,211668.000000


In [29]:
flag_mid = {
    "A":10,
    "B":60,
    "C":175,
    "E":375,
    "F":750,
    "G":1750,
    "H":3750,
    "I":7500,
    "J":17500,
    "K":37500,
    "L":75000,
    "M":100000
}
def unify_cbp_emp(df):
    df["emp"] = pd.to_numeric(df["emp"], errors="coerce")
    
    if "emp_nf" in df.columns:
        df["emp_nf"] = pd.to_numeric(df["emp_nf"], errors="coerce")
        df["emp"] = df["emp"].fillna(df["emp_nf"])
    
    elif "empflag" in df.columns:
        mask = (df["emp"].isna()) | ((df["emp"] == 0) & df["empflag"].notna())
        df.loc[mask, "emp"] = df.loc[mask, "empflag"].map(flag_mid)

    df = df[df["emp"].notna()]
    
    return df

In [30]:
df = unify_cbp_emp(df)

In [31]:
df['naics'] = df["naics"].astype(str)

In [32]:
cbp_manuf = df[df["naics"].str[:2].isin(["31","32","33"])].copy()

cbp_non_manuf = df[~df["naics"].str[:2].isin(["31","32","33"])].copy()

In [33]:
# --- Manufacturing ---
cz_ind_manuf = (
    cbp_manuf
    .groupby(["CZ","naics"], as_index=False)["emp"]
    .sum()
)

cz_total_manuf = (
    cz_ind_manuf
    .groupby(["CZ"], as_index=False)["emp"]
    .sum()
    .rename(columns={"emp": "L_manuf"})
)

# --- Non-Manufacturing ---
cz_ind_non_manuf = (
    cbp_non_manuf
    .groupby(["CZ","naics"], as_index=False)["emp"]
    .sum()
)

cz_total_non_manuf = (
    cz_ind_non_manuf
    .groupby(["CZ"], as_index=False)["emp"]
    .sum()
    .rename(columns={"emp": "L_non_manuf"})
)

In [34]:
cz_ind_manuf.to_csv(f"./{year}/cz_industry_manuf_{year}.csv", index=False)
cz_total_manuf.to_csv(f"./{year}/cz_total_manuf_{year}.csv", index=False)

cz_ind_non_manuf.to_csv(f"./{year}/cz_industry_non_manuf_{year}.csv", index=False)
cz_total_non_manuf.to_csv(f"./{year}/cz_total_non_manuf_{year}.csv", index=False)

OSError: Cannot save file into a non-existent directory: '1999'